# Chapter 37 — Capstone implementation: load, filter, table, KPIs, chart, and export

**Book role:** Capstone  
**Suggested book space:** about 12 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Compose service functions into a complete Tkinter data workflow.
- Render filters, KPIs, Treeview results, and a Matplotlib chart.
- Use file dialogs and safe export semantics.
- Make refreshes reproducible and failures recoverable.

## 1. Start with the data service

Keep the analytics pipeline independent of the Tkinter application. The service should load a selected file, normalize column names, validate required columns, derive numerical measures, apply filters, and return a well-defined result object. Do not mutate a shared raw DataFrame silently; retain raw/cleaned separation if the use case requires auditing. The UI should receive both a filtered DataFrame and a summary/quality report.

The service should have deterministic behavior on the course’s synthetic dataset. Treat file read errors, schema errors, and row-level data-quality warnings as different outcomes. A valid file with a few flagged rows may be loadable after the user accepts warnings; a file missing the required order ID or revenue inputs may need to be rejected. Define severity and handling before wiring the GUI.

### Worked example 1: Create reusable sales transformations

This is a baseline, not a complete policy for all edge cases. Add row-level quality issues for coercion failures, status rules, unknown categories, and profit consistency. A production import should preserve the original raw data when auditing is required.

In [ ]:
import pandas as pd

REQUIRED = {"order_id", "order_date", "customer_id", "region", "product", "category", "quantity", "unit_price", "unit_cost", "status"}

def normalize_sales(raw: pd.DataFrame) -> pd.DataFrame:
    frame = raw.copy()
    frame.columns = [str(column).strip().lower().replace(" ", "_") for column in frame.columns]
    missing = REQUIRED - set(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    frame["order_date"] = pd.to_datetime(frame["order_date"], errors="coerce")
    for column in ("quantity", "unit_price", "unit_cost"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame["revenue"] = frame["quantity"] * frame["unit_price"]
    frame["cost"] = frame["quantity"] * frame["unit_cost"]
    frame["profit"] = frame["revenue"] - frame["cost"]
    frame["profit_margin"] = frame["profit"] / frame["revenue"].where(frame["revenue"].ne(0))
    return frame

# Example with the course pack: raw = pd.read_csv("data/sample_sales.csv"); clean = normalize_sales(raw)

## 2. Build the UI as a composition of small views

The root application can contain a toolbar, a filter frame, KPI cards, a Notebook with Overview/Data/Quality tabs, and a status bar. Use `ttk` controls for native behavior, `Treeview` for a bounded detail page, and `FigureCanvasTkAgg` for Matplotlib. Create each region in a dedicated method or view class so that layout and rendering can be changed without rewriting data calculations.

A filter Apply action should: collect current selections; validate them; ask the service to compute a result; render the KPI set, current table page, chart, and quality summary; and update the status with row counts and refresh timestamp. If a refresh fails, preserve the last valid view with a stale/error indicator unless the product requirements say otherwise. Do not display a success message before the service returns success.

### Worked example 2: Build filtered data and a summary in one service

This makes the filter and KPI semantics testable. For date filters, make the UI show that end date is exclusive internally or convert the visible inclusive date to the next day deliberately.

In [ ]:
from dataclasses import dataclass
import pandas as pd

@dataclass(frozen=True)
class SalesFilters:
    start_date: pd.Timestamp | None = None
    end_date: pd.Timestamp | None = None
    region: str = "All"
    category: str = "All"
    status: str = "Paid"


def query_sales(frame: pd.DataFrame, filters: SalesFilters) -> tuple[pd.DataFrame, dict]:
    result = frame.copy()
    if filters.start_date is not None:
        result = result[result["order_date"] >= filters.start_date]
    if filters.end_date is not None:
        result = result[result["order_date"] < filters.end_date]
    if filters.region != "All":
        result = result[result["region"].eq(filters.region)]
    if filters.category != "All":
        result = result[result["category"].eq(filters.category)]
    if filters.status != "All statuses":
        result = result[result["status"].eq(filters.status)]
    revenue = float(result["revenue"].sum())
    profit = float(result["profit"].sum())
    summary = {"line_count": len(result), "orders": int(result["order_id"].nunique()),
               "customers": int(result["customer_id"].nunique()), "revenue": revenue,
               "profit": profit, "margin": profit / revenue if revenue else None}
    return result, summary

## 3. Make export scope explicit

Provide separate actions for “Export filtered rows” and “Export complete cleaned dataset” only if both are needed. For a multi-sheet Excel report, include a summary, filtered data, data-quality issues, and data dictionary. Include the filters/date range and generated timestamp so recipients know what was included.

Use a temporary output path for critical exports and replace the target only after the workbook is written and validated. Confirm overwrite when the selected destination already exists. Handle permission and disk failures without deleting the previous good report. If formula-like untrusted text is present, apply and document the spreadsheet-safe export policy from the security chapter.

## Guided lab: Lab: implement the MVP

Implement the core workbench in `examples/`: Open CSV/Excel, validate and normalize, show filter controls, display KPIs, render a monthly trend, paginate/sort a Treeview, display data-quality findings, and export CSV/Excel. Add loading/empty/error states and protect the previous valid dataset from failed imports. Write tests for calculations before polishing layout.

## Review questions and practice

1. Which operations belong in the data service instead of a widget callback?
2. Why should a failed import preserve the last valid dataset?
3. What should a multi-sheet report contain for reproducibility?
4. How will you make the table and chart reflect the same filters?
5. Which capstone features should be implemented before database connectors?

## Further reading

- pandas I/O: https://pandas.pydata.org/docs/user_guide/io.html
- Matplotlib Tk embedding: https://matplotlib.org/stable/gallery/user_interfaces/embedding_in_tk_sgskip.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.